# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly
# only build with ONNX gpu + caspar + ceres cuda/cudss; downloads disabled, so model paths must be set
!pip uninstall -y --quiet pycolmap
!pip install --progress-bar off --quiet \
   "https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl"

In [ ]:
from tqdm import tqdm
from pathlib import Path
import json
import numpy as np
import os
import shutil
import urllib.request


from hloc import reconstruction, visualization
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d
import pycolmap

In [4]:
DATA_PATH = Path('/kaggle/input/datasets/yu5uf5/buggy-hloc')
outputs = Path('/kaggle/working/multi')
outputs.mkdir(exist_ok=True)
IMAGES_PATH = Path('/tmp/images')

In [ ]:
# camera position relative to the racebox in body frame, meters: {roll_id: (forward, left)}
# +forward = camera ahead of the racebox, +left = camera to its left
RB_CAM_OFFSET = {
    37: (1.25, 0), 38: (1.25, 0),
    39: (1.20, 0),
    44: (-0.80, 0), 45: (-0.80, 0),
    1387: (0.05, 0),
    1388: (-0.75, 0),
    1401: (-0.70, 0)
}

R_EARTH = 6371000.0

def apply_rb_cam_offset(gps, fwd, left):
    """Move racebox positions to the camera using gps-derived heading."""
    lat = np.array([s['lat'] for s in gps], float)
    lon = np.array([s['long'] for s in gps], float)
    latr = np.radians(lat)
    i = np.arange(len(gps))
    i0, i1 = np.maximum(i - 12, 0), np.minimum(i + 12, len(gps) - 1)  # ~0.5 s window at 25 Hz
    dn = np.radians(lat[i1] - lat[i0]) * R_EARTH
    de = np.radians(lon[i1] - lon[i0]) * R_EARTH * np.cos(latr)
    ok = np.hypot(de, dn) > 0.5
    if not ok.any():
        return
    head = np.arctan2(de, dn)
    last = np.maximum.accumulate(np.where(ok, i, -1))
    last[last < 0] = np.flatnonzero(ok)[0]  # hold heading through standstill
    head = head[last]
    off_e = fwd * np.sin(head) - left * np.cos(head)
    off_n = fwd * np.cos(head) + left * np.sin(head)
    for s, oe, on, phi in zip(gps, off_e, off_n, latr):
        s['lat'] += np.degrees(on / R_EARTH)
        s['long'] += np.degrees(oe / (R_EARTH * np.cos(phi)))

In [5]:

videos = {v.stem: v for v in list(DATA_PATH.glob('*.mp4'))}
data = {}
for p in DATA_PATH.glob('vid_imu/*.json'):
    with open(p) as f:
        data[p.stem] = json.load(f)
# shift racebox streams onto the virb timeline (imu-estimated in smooth.ipynb export)
for d in data.values():
    off_ns = d.get('racebox_offset_ms', 0.0) * 1e6
    for key in ('racebox_gps', 'racebox_speed'):
        for s in d.get(key, []):
            s['timestamp'] += off_ns

# move racebox gps to the camera position (measured per-roll mounting offsets)
for run, d in data.items():
    fwd, left = RB_CAM_OFFSET.get(int(run), (0.0, 0.0))
    if fwd or left:
        apply_rb_cam_offset(d.get('racebox_gps', []), fwd, left)


# Create Images

In [ ]:
import cv2

STRIDE = 1  # keep every Nth frame
sharpness = {}  # run -> {ts_ns: variance of laplacian}; persisted as <images>/<run>/sharpness.json

def sharpness_score(bgr):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())

In [ ]:
for stem, video in tqdm(videos.items(), desc='videos'):
    start_ns = data[stem]['camera_start']  # video start, nanoseconds
    out_dir = IMAGES_PATH / stem
    if (out_dir / 'sharpness.json').exists():
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    sharpness[stem] = {}
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved = 0
    i = 0
    try:
        with tqdm(total=frame_count, desc=stem, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % STRIDE != 0:
                    continue
                timestamp_ns = start_ns + int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1_000_000))
                cv2.imwrite(str(out_dir / f"{timestamp_ns}.jpg"), frame)
                sharpness[stem][timestamp_ns] = sharpness_score(frame)
                saved += 1
    finally:
        cap.release()
    with open(out_dir / 'sharpness.json', 'w') as f:
        json.dump(sharpness[stem], f)
    print(f"{stem}: saved {saved} frames to {out_dir}")

# Config

In [ ]:
sfm_pairs = outputs / 'pairs-sfm.txt'
loc_pairs = outputs / 'pairs-loc.txt'
sfm_dir = outputs / 'sfm'
sfm_prior_dir = outputs / 'sfm_prior'
database = outputs / 'database.db'

RESUME = False     # continue from outputs/database.db + outputs/prior
RECON_RUNS = None  # runs to reconstruct this pass; None = all

In [ ]:
# racebox: 25Hz + scalar doppler speed; fit: 10Hz + velocity vectors
GPS_KIND = 'racebox'
LAT0, LON0, ALT0 = 40.44163016, -79.94165829, 288.42151354  # shared ENU reference

DELTA_S = 2.0         # m between selected frames
SEQ_K = 6             # forward sequential pairs per frame
CROSS_K = 3           # nearest cross-run candidates per frame
CROSS_R = 6.0         # m, max cross-run pair distance
HEADING_MAX_DEG = 40  # max cross-run heading difference

CAM_HEIGHT = 0.35  # m, gps alt is DEM road level; lift priors approximatley to the camera

In [ ]:
import torch

GPU_INDEX = ','.join(str(i) for i in range(torch.cuda.device_count()))

# caspar BA only supports SIMPLE_RADIAL [f, cx, cy, k1]/PINHOLE
# initial estimates refined per image
CAMERA_PARAMS = [653.4, 631.72, 338.74, -0.0526]

# Mapping

## Select Frames

In [ ]:
runs = sorted(data, key=int)
gps_field = 'racebox_gps' if GPS_KIND == 'racebox' else 'gps_data'
gt = pycolmap.GPSTransform(pycolmap.GPSTransfromEllipsoid.WGS84)

def gps_arrays(run):
    gps = data[run][gps_field]
    return tuple(np.array([s[k] for s in gps], float) for k in ('timestamp', 'lat', 'long', 'alt'))

def speed_arrays(run):
    if GPS_KIND == 'racebox':
        spd = data[run]['racebox_speed']
        return (np.array([s['timestamp'] for s in spd], float),
                np.array([s['speed'] for s in spd], float))
    vel = data[run]['velocity']
    return (np.array([s['timestamp'] for s in vel], float),
            np.hypot([s['vx'] for s in vel], [s['vy'] for s in vel]))

sel = {}
run_enu = {}
for run in runs:
    image_paths = sorted((IMAGES_PATH / run).glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    gps_ts, lat, lon, alt = gps_arrays(run)
    enu_src = np.array(gt.ellipsoid_to_enu(list(np.stack([lat, lon, alt], 1)), LAT0, LON0, ALT0))
    enu_src[:, 2] += CAM_HEIGHT
    run_enu[run] = (gps_ts, enu_src)
    spd_ts, spd = speed_arrays(run)

    # distance-uniform selection over the video ∩ gps window
    m = (spd_ts >= max(gps_ts[0], image_ts[0])) & (spd_ts <= min(gps_ts[-1], image_ts[-1]))
    ts_w, v_w = spd_ts[m], spd[m]
    dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
    sel_ts = np.interp(np.arange(0.0, dist[-1], DELTA_S), dist, ts_w)

    # sharpest frame within each target's window (blur is vibration-driven, varies frame to frame)
    score_file = IMAGES_PATH / run / 'sharpness.json'
    if run not in sharpness and score_file.exists():
        with open(score_file) as f:
            sharpness[run] = {int(k): v for k, v in json.load(f).items()}
    scores = sharpness.get(run)
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2],
                             (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
        elif scores:
            idx.append(i0 + int(np.argmax([scores.get(int(u), 0.0) for u in image_ts[i0:i1]])))
        else:
            idx.append(i0 + np.abs(image_ts[i0:i1] - t).argmin())
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, gps_ts, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    sel[run] = {
        'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
        'enu': enu,
        'heading': np.arctan2(grad[:, 1], grad[:, 0]),
    }
    print(f'{run}: {len(idx)} frames over {dist[-1]:.0f}m')

# selection must reproduce db names (guards param drift across sessions)
if RESUME:
    with pycolmap.Database.open(str(database)) as db:
        db_names = {im.name for im in db.read_all_images()}
    for run in runs:
        mine = {n for n in db_names if n.startswith(f'{run}/')}
        assert not mine or mine == set(sel[run]['names']), run

references = [n for run in runs for n in sel[run]['names']]
recon_names = sorted(n for r in (RECON_RUNS or runs) for n in sel[r]['names'])
len(references)

In [ ]:
import matplotlib.pyplot as plt

for run in runs:
    plt.plot(*sel[run]['enu'][:, :2].T, '.', ms=2, label=run)
plt.axis('equal')
plt.legend(markerscale=5)
plt.show()

In [ ]:
sl = slice(200, 210)
plot_images([read_image(IMAGES_PATH / ref) for ref in references[sl]],
            titles=references[sl], dpi=25)

## Features

In [ ]:
# this pycolmap build can't auto-download onnx models; stage them locally
MODELS_PATH = Path('/kaggle/working/models')
MODELS_PATH.mkdir(exist_ok=True)
for name in ('aliked-n16rot.onnx', 'aliked-lightglue.onnx'):
    f = MODELS_PATH / name
    if not f.exists():
        urllib.request.urlretrieve(
            f'https://github.com/colmap/colmap/releases/download/3.13.0/{name}', f)


# colmap masks: <mask_path>/<image name>.png, black = exclude; all frames share mask0
MASKS_PATH = Path('/kaggle/working/masks')
MASKS_PATH.mkdir(exist_ok=True)
mask_src = MASKS_PATH / 'mask0.png'
if not mask_src.exists():
    shutil.copy(DATA_PATH / 'mask0.png', mask_src)
for ref in references:
    dst = MASKS_PATH / f'{ref}.png'
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        os.link(mask_src, dst)

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, outputs / 'colmap.LOG.')
if not RESUME:
    database.unlink(missing_ok=True)

extraction_options = pycolmap.FeatureExtractionOptions()
extraction_options.type = pycolmap.FeatureExtractorType.ALIKED_N16ROT
extraction_options.aliked.n16rot_model_path = str(MODELS_PATH / 'aliked-n16rot.onnx')
extraction_options.aliked.max_num_features = 4096
extraction_options.use_gpu = True
extraction_options.gpu_index = GPU_INDEX

pycolmap.extract_features(
    database, IMAGES_PATH, image_names=sorted(references),
    camera_mode=pycolmap.CameraMode.PER_FOLDER,  # consider PER_IMAGE because of stabilization
    reader_options={'camera_model': 'SIMPLE_RADIAL',
                    'camera_params': ','.join(str(v) for v in CAMERA_PARAMS),
                    'mask_path': str(MASKS_PATH)},
    extraction_options=extraction_options)

## Matching

In [ ]:
from itertools import combinations
from scipy.spatial import cKDTree

pairs = set()
for run in runs:
    names = sel[run]['names']
    for i in range(len(names)):
        for j in range(i + 1, min(i + 1 + SEQ_K, len(names))):
            pairs.add((names[i], names[j]))
n_seq = len(pairs)

for a, b in combinations(runs, 2):
    tree = cKDTree(sel[b]['enu'][:, :2])
    dists, nbrs = tree.query(sel[a]['enu'][:, :2], k=CROSS_K, distance_upper_bound=CROSS_R)
    for i, (ds, js) in enumerate(zip(dists, nbrs)):
        for d, j in zip(ds, js):
            if not np.isfinite(d):
                continue
            dh = abs(sel[a]['heading'][i] - sel[b]['heading'][j])
            if min(dh, 2 * np.pi - dh) <= np.radians(HEADING_MAX_DEG):
                pairs.add(tuple(sorted((sel[a]['names'][i], sel[b]['names'][j]))))

sfm_pairs.write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(f'{n_seq} sequential + {len(pairs) - n_seq} cross-run pairs')

In [ ]:
matching_options = pycolmap.FeatureMatchingOptions()
matching_options.type = pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE
matching_options.aliked.lightglue.model_path = str(MODELS_PATH / 'aliked-lightglue.onnx')
matching_options.use_gpu = True
matching_options.gpu_index = GPU_INDEX

pairing_options = pycolmap.ImportedPairingOptions()
pairing_options.match_list_path = str(sfm_pairs)

pycolmap.match_image_pairs(database, matching_options=matching_options,
                           pairing_options=pairing_options)

## Reconstruct

### Database

In [ ]:
PRIOR_STD_XY = 0.5
PRIOR_STD_Z = 1.0
cov = np.diag([PRIOR_STD_XY**2, PRIOR_STD_XY**2, PRIOR_STD_Z**2])

# inert unless use_prior_position is set, so both reconstructions can share the db
with pycolmap.Database.open(str(database)) as db:
    have = {p.corr_data_id.id for p in db.read_all_pose_priors()}
    for image in db.read_all_images():
        if image.data_id.id in have:
            continue
        p = Path(image.name)
        gps_ts, enu = run_enu[p.parts[0]]
        ts = int(p.stem)
        prior = pycolmap.PosePrior()
        prior.corr_data_id = image.data_id
        prior.position = np.array([np.interp(ts, gps_ts, enu[:, i]) for i in range(3)])
        prior.position_covariance = cov
        prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
        db.write_pose_prior(prior)
    print('pose priors:', db.num_pose_priors())

### No Prior

In [ ]:
# model = reconstruction.run_reconstruction(
#     sfm_dir, database, IMAGES_PATH, verbose=True,
#     options={
#         "image_names": recon_names,
#         "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#         "ba_global_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#     })

In [ ]:
# p = outputs / 'no_prior'
# p.mkdir(parents=True, exist_ok=True)
# model.write(p)

### Prior

In [ ]:
# priors only constrain global BA, and caspar doesn't support them
prior_options = {
    "image_names": recon_names,
    "use_prior_position": True,
    # "use_robust_loss_on_prior_position": True,
    # Speed options
    # "ba_use_gpu": True,
    # "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
    # "ba_global_backend": pycolmap.BundleAdjustmentBackend.CERES,
    # "ba_global_frames_ratio": 1.3,
    # "ba_global_points_ratio": 1.3,
    # "ba_local_max_num_iterations": 12,
    # "ba_local_max_refinements": 2,
    # "ba_global_max_num_iterations": 30,
    # "ba_global_max_refinements": 3,
    # "mapper": {"ba_global_ignore_redundant_points3D": True},
}
if RESUME:
    shutil.rmtree(sfm_prior_dir, ignore_errors=True)
    sfm_prior_dir.mkdir(parents=True)
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(
            database, IMAGES_PATH, sfm_prior_dir,
            options={**prior_options, "fix_existing_frames": False},
            input_path=str(outputs / 'prior'))
    model_prior = recs[0]
else:
    model_prior = reconstruction.run_reconstruction(
        sfm_prior_dir, database, IMAGES_PATH, verbose=True, options=prior_options)

In [ ]:
p = outputs / 'prior'
p.mkdir(parents=True, exist_ok=True)
model_prior.write(p)

# Visualize

### No Prior

In [13]:
# model = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'no_prior'))

In [15]:
# fig = viz_3d.init_figure()
# viz_3d.plot_reconstruction(fig, model, points_rgb=True)
# fig.show()

### Prior

In [ ]:
model_prior = pycolmap.Reconstruction(str(outputs / 'prior'))

In [ ]:
with pycolmap.Database.open(str(database)) as db:
    priors = db.read_all_pose_priors()
enu = {p.corr_data_id.id: p.position for p in priors}  # already ENU
errs = np.array([model_prior.images[i].projection_center() - enu[i]
                 for i in model_prior.reg_image_ids()])
print(f"rmse vs gps: {np.sqrt((errs[:, :2] ** 2).sum(1).mean()):.2f} m horizontal, "
      f"{np.sqrt((errs[:, 2] ** 2).mean()):.2f} m vertical")

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model_prior, points_rgb=True)
fig.show()